# 01 · Pretraining the Mini GPT (Colab, T4 GPU)

Trains the decoder-only Transformer from random weights on the Tidewell waste-guide corpus (next-token prediction).

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Everything is saved to Google Drive under `MyDrive/mini-gpt/`, so if the session disconnects, just run all cells again: training **resumes** from the last checkpoint.

📸 = take a screenshot of that cell's output for the report.

In [ ]:
# 📸 GPU used for training (report: hardware)
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/mini-gpt'
RUN_DIR = f'{DRIVE}/pretrain'
RESULTS = f'{DRIVE}/results'
import os
os.makedirs(RUN_DIR, exist_ok=True)
for sub in ('figures', 'logs', 'samples', 'tables'):
    os.makedirs(f'{RESULTS}/{sub}', exist_ok=True)

In [ ]:
# Get the latest code + data from GitHub
REPO = 'https://github.com/Shayan047/mini-gpt.git'
if os.path.exists('/content/mini-gpt'):
    !git -C /content/mini-gpt pull
else:
    !git clone {REPO} /content/mini-gpt
%cd /content/mini-gpt
!git log --oneline -1

In [ ]:
# Sanity checks: tokenizer + model tests (causal mask, shapes, initial loss, overfitting one batch)
!python -m tests.test_tokenizer
!python -m tests.test_model

In [ ]:
# 📸 Model architecture and parameter count (report: architecture)
import json, torch
from src.model import GPT, GPTConfig
meta = json.load(open('data/tokens/meta.json'))
cfg = GPTConfig(vocab_size=meta['vocab_size'])
model = GPT(cfg)
print(cfg)
for k, v in model.num_params(by_component=True).items():
    print(f'{k:55s} {v:>12,}')
print(meta['splits'])
del model

## Train
Default: 6 layers, 6 heads, d_model 384, context 256, batch 32, 3,000 steps (~20 epochs), AdamW lr 6e-4 with warmup + cosine decay, fp16 mixed precision.
Expect roughly 15–30 minutes on a T4. Re-running this cell after a disconnect resumes from `ckpt_last.pt`.

In [ ]:
# 📸 The last ~15 lines of this output (training log)
!python -m src.train --out-dir "{RUN_DIR}"

In [ ]:
# 📸 Loss curves (report: training procedure / results)
!python -m src.plot_losses --log "{RUN_DIR}/log.csv" --out "{RESULTS}/figures/pretrain_loss.png" --title "Pretraining: next-token prediction"
from IPython.display import Image, display
display(Image(f'{RESULTS}/figures/pretrain_loss.png'))

In [ ]:
# How generations from fixed prompts changed during training (first vs last evaluation)
import json
samples = [json.loads(l) for l in open(f'{RUN_DIR}/samples.jsonl')]
steps = sorted({s['step'] for s in samples})
for step in (steps[0], steps[len(steps)//2], steps[-1]):
    print(f'================ step {step} ================')
    for s in samples:
        if s['step'] == step and s['temperature'] == 0.0:
            print(repr(s['text'][:200]))

In [ ]:
# Copy logs/samples into the Drive results folder -> download it into the repo's results/ folder
import shutil
for f in ('log.csv', 'train_log.txt', 'summary.json'):
    shutil.copy(f'{RUN_DIR}/{f}', f'{RESULTS}/logs/pretrain_{f}')
shutil.copy(f'{RUN_DIR}/samples.jsonl', f'{RESULTS}/samples/pretrain_samples_during_training.jsonl')
print(open(f'{RUN_DIR}/summary.json').read())
!ls -la "{RUN_DIR}" "{RUN_DIR}/snapshots"